# SI4006 · Sesión 2 — Lab: Abrir la caja

### Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT · Semana 2

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<<<ORG>>>/<<<REPO>>>/blob/main/sesiones/s02/S02_Lab_Abrir_la_caja.ipynb)

---

**Hoy no miran. Hacen.**

En la Sesión 1 les pedí explícitamente que *no* entendieran el código. Hoy es lo contrario: al final de estas tres horas van a haber escrito, con sus manos, el mecanismo que hace funcionar a todos los modelos que usamos la semana pasada.

Tres laboratorios:

| | Lab | Qué van a descubrir |
|---|---|---|
| **A** | Forense de tokenizadores | Que su apellido es caro, y que hablar español cuesta más plata |
| **B** | Attention a mano | Que la atención es un promedio ponderado, y nada más |
| **C** | La atención no sabe de orden | Que sin ayuda, un transformer no distingue "el perro mordió al hombre" de "el hombre mordió al perro" |

> ⚙️ **No necesitan GPU hoy.** Todo corre en CPU. `Runtime → Change runtime type → CPU` está bien.
>
> 🔄 **Puntos de sincronización:** al final de cada lab hay una parada. Nadie sigue hasta que todos lleguen.
>
> 🛟 Cada experimento tiene, al final, el resultado esperado en una celda plegable. Si algo falla, la conversación sigue.

## 0 · Setup

> ⚠️ **Lección de la Sesión 1:** no fijamos versiones a ciegas. Primero miramos qué trae Colab, y solo instalamos si falta algo.

In [ ]:
# Diagnóstico: qué tenemos ANTES de tocar nada
import importlib.metadata as md_, sys

print('python', sys.version.split()[0])
for p in ['torch', 'transformers', 'tokenizers', 'sentencepiece', 'pandas']:
    try:
        print(f'{p:16} {md_.version(p)}')
    except md_.PackageNotFoundError:
        print(f'{p:16} FALTA')

In [ ]:
# Solo instala lo que falte. Si el diagnóstico salió limpio, esta celda no hace casi nada.
try:
    import transformers, sentencepiece  # noqa: F401
    print('Entorno completo — no instalo nada.')
except ImportError:
    %pip -q install transformers sentencepiece
    print('Instalado. Si Colab pide RESTART RUNTIME, reinicia y vuelve a correr desde aquí.')

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer

torch.manual_seed(42)
pd.set_option('display.max_colwidth', None)
print('torch', torch.__version__, '| listo')

---
# LAB A · Forense de tokenizadores
### ⏱ 25 minutos

Un modelo no lee texto. Lee **números**. El tokenizador es el traductor, y como todo traductor, tiene opiniones.

Vamos a cargar cinco tokenizadores de tres familias distintas y a ponerlos a pelear sobre el mismo texto.

In [ ]:
# Cinco tokenizadores, tres familias. Descarga ~1 min la primera vez.
TOKENIZADORES = {
    'GPT-2 (BPE)':            'gpt2',
    'BERT-en (WordPiece)':    'bert-base-uncased',
    'BERT-multi (WordPiece)': 'bert-base-multilingual-cased',
    'XLM-R (SentencePiece)':  'xlm-roberta-base',
    'Qwen2.5 (BPE moderno)':  'Qwen/Qwen2.5-3B-Instruct',
}

toks = {}
for nombre, ruta in TOKENIZADORES.items():
    toks[nombre] = AutoTokenizer.from_pretrained(ruta)
    print(f'{nombre:24} vocabulario: {toks[nombre].vocab_size:>7,} tokens')

> 🙋 **Primera pregunta, antes de seguir:** miren los tamaños de vocabulario. Van desde ~30.000 hasta más de 250.000.
>
> Un vocabulario grande significa menos tokens por frase (más barato de correr) pero una tabla de embeddings enorme (más memoria). Es el primer trade-off del día y no tiene respuesta correcta.

In [ ]:
def comparar(texto):
    """Muestra cómo cada tokenizador despedaza el mismo texto."""
    filas = []
    for nombre, tk in toks.items():
        piezas = tk.tokenize(texto)
        filas.append({'Tokenizador': nombre, 'N': len(piezas), 'Tokens': ' | '.join(piezas)})
    return pd.DataFrame(filas).set_index('Tokenizador')

comparar('Inteligencia artificial')

> 👀 **Miren los símbolos raros.** No son basura: son la firma de cada familia.
>
> - `##` → WordPiece. Marca "esto va pegado a lo anterior".
> - `Ġ` → BPE de GPT-2. Marca "aquí había un espacio antes".
> - `▁` → SentencePiece. Mismo trabajo, otro símbolo.
>
> Las tres familias resuelven el mismo problema —¿cómo represento el espacio en blanco?— y las tres lo resolvieron distinto. Con eso ya pueden identificar de qué familia es un tokenizador con solo mirar su salida.

### Experimento 1 · Sus apellidos

Van a medir cuánto cuesta su propio nombre.

In [ ]:
NOMBRES = ['Smith', 'Sepúlveda', 'Idárraga', 'Castañeda', 'Saldarriaga',
           'Bustamante', 'Citelly', 'Higuita', 'Arbeláez']

filas = []
for n in NOMBRES:
    fila = {'Nombre': n}
    for nombre, tk in toks.items():
        fila[nombre.split(' ')[0]] = len(tk.tokenize(n))
    filas.append(fila)

pd.DataFrame(filas).set_index('Nombre')

In [ ]:
# Miren el destrozo de cerca
comparar('Sepúlveda')

> 🙋 **Al grupo:** ¿cuántos tokens cuesta `Smith`? ¿Cuántos cuesta `Sepúlveda`?
>
> Ninguno de los dos es más complejo que el otro como palabra. La diferencia no es lingüística: es que el corpus de entrenamiento tenía muchos más Smith que Sepúlveda, y el tokenizador aprendió a darle una pieza propia a lo frecuente.
>
> **Su apellido es caro porque el internet en el que se entrenó esto no repetia apellidos colombianos.**
>
> 🔍 **Y miren de cerca la salida de BERT-en:** ¿sobrevivió la tilde de `Sepúlveda`? ¿Sobrevivió la `ñ` de `Castañeda`? Ese tokenizador no solo fragmenta el español: hay un paso de normalización que puede estar borrando información antes de que el modelo vea nada. Nadie lo decidió con mala intención — alguien asumió que el texto venía en inglés.

<details><summary>🛟 Resultado esperado</summary>

`Smith` suele costar 1 token en casi todos. `Sepúlveda`, `Idárraga` y `Saldarriaga` se fragmentan en 4–6 piezas en los tokenizadores centrados en inglés, y bastante menos en los multilingües (XLM-R, BERT-multi, Qwen). En BERT-en, además, es probable que las tildes y la `ñ` desaparezcan por completo.
</details>

### Experimento 2 · El impuesto lingüístico

La misma idea, dicha en dos idiomas. ¿Cuesta lo mismo?

In [ ]:
PARES = [
    ('The cat is sleeping on the sofa.',              'El gato está durmiendo en el sofá.'),
    ('Artificial intelligence is changing the world.', 'La inteligencia artificial está cambiando el mundo.'),
    ('I need to talk to a lawyer about my contract.',  'Necesito hablar con un abogado sobre mi contrato.'),
]

filas = []
for en, es in PARES:
    for nombre, tk in toks.items():
        n_en, n_es = len(tk.tokenize(en)), len(tk.tokenize(es))
        filas.append({'Tokenizador': nombre.split(' ')[0], 'Frase': en[:28] + '…',
                      'EN': n_en, 'ES': n_es, 'Sobrecosto ES': f'{(n_es/n_en - 1)*100:+.0f}%'})

pd.DataFrame(filas).pivot_table(index='Frase', columns='Tokenizador',
                                values=['EN', 'ES'], aggfunc='first')

In [ ]:
# El promedio, que es el número que importa
for nombre, tk in toks.items():
    r = [len(tk.tokenize(es)) / len(tk.tokenize(en)) for en, es in PARES]
    print(f'{nombre:24} el español cuesta {sum(r)/len(r):.2f}x lo que el inglés')

> 💸 **Esto no es trivia académica. Es una factura.**
>
> Las APIs de LLM cobran por token. Si su sistema responde en español, están pagando un sobrecosto por decir exactamente lo mismo. La ventana de contexto también se les llena más rápido: el "límite de 8.000 tokens" es más corto en español que en inglés.
>
> Hay un sesgo económico incrustado en la infraestructura, y no lo puso nadie a propósito. Salió de qué idioma tenía más texto en internet cuando se entrenó el tokenizador.
>
> **Guarden esto para el Módulo 5**, cuando hablemos de costo de inferencia. Y para sus proyectos: todos son en español.

<details><summary>🛟 Resultado esperado</summary>

Los tokenizadores centrados en inglés (GPT-2, BERT-en) muestran sobrecostos del 30–80% para el español. Los multilingües (XLM-R, Qwen, BERT-multi) lo reducen bastante, pero rara vez llegan a la paridad.
</details>

### Experimento 3 · Los números

Por qué estos modelos son malos para la aritmética.

In [ ]:
for n in ['7', '42', '1234', '1234567', '3.14159']:
    print(f'\n{n!r}')
    for nombre, tk in toks.items():
        print(f'   {nombre.split(" ")[0]:12} → {tk.tokenize(n)}')

> 🧮 **Miren `1234567`.** Para varios tokenizadores no es un número: es un puñado de fragmentos arbitrarios, partidos donde el algoritmo encontró frecuencias, no donde están las unidades, decenas y centenas.
>
> Ahora pregúntense: ¿cómo va a sumar bien un modelo que ni siquiera ve los dígitos alineados? No es que "no sepa matemáticas". Es que **le entregamos el problema ya destrozado**, antes de que el modelo vea nada.
>
> Muchas de las cosas que la gente atribuye a que "el modelo es tonto" son en realidad decisiones de tokenización tomadas años antes.

### Experimento 4 · Su dominio  ✍️

Ahora ustedes.

**TODO:** reemplacen el texto de abajo por una frase real de su dominio — un artículo de ley, un nombre de fármaco, un término agro, una descripción de producto. Algo que su sistema va a tener que procesar de verdad este semestre.

In [ ]:
# TODO — reemplacen esto por texto real de SU dominio
MI_TEXTO = 'El fuero sindical protege al trabajador aforado.'   # ← cámbienlo

comparar(MI_TEXTO)

> 🙋 **Al grupo, en voz alta:** ¿a quién le destrozó el tokenizador un término clave de su dominio?
>
> Si la jerga central de su proyecto se fragmenta en cinco pedazos, acaban de encontrar un problema real — y todavía no han entrenado nada. Anótenlo: en la Sesión 4, cuando escojan su modelo base, este va a ser uno de los criterios de decisión.

---
### 🔄 PUNTO DE SINCRONIZACIÓN — fin del Lab A

Antes de seguir, deberían poder responder tres cosas sin mirar:

1. ¿Cuál es la firma de WordPiece, de BPE y de SentencePiece?
2. ¿Por qué su apellido cuesta más tokens que `Smith`?
3. ¿Por qué es más caro operar un chatbot en español?

---
# LAB B · Attention a mano
### ⏱ 35 minutos

Acaban de ver cómo el texto se convierte en tokens, y los tokens en vectores.

Ahora viene la pregunta del curso: **¿qué hace el transformer con esos vectores?**

La respuesta cabe en tres líneas de código. Las van a escribir ustedes.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Tres pasos, y cada uno tiene una razón de ser:

1. **$QK^\top$** — cada token le pregunta a cada token: *¿qué tan relevante eres para mí?* Es un producto punto, o sea, una similitud. **La misma operación de `doc_emb @ q.T` de la Sesión 1.**
2. **$\div \sqrt{d_k}$** — sin esto los números crecen con la dimensión, el softmax se satura y los gradientes mueren. Es plomería, pero es plomería que hace la diferencia entre entrenar y no entrenar.
3. **$\text{softmax}(\cdot)V$** — se convierten en pesos que suman 1, y se usan para promediar los valores.

**La atención es un promedio ponderado donde los pesos los decide una similitud. Eso es todo.**

In [ ]:
# Escenario de juguete: 4 tokens, 8 dimensiones
L, D = 4, 8
Q = torch.randn(1, L, D)   # (batch, tokens, dim)
K = torch.randn(1, L, D)
V = torch.randn(1, L, D)
print('Q:', tuple(Q.shape), '| K:', tuple(K.shape), '| V:', tuple(V.shape))

In [ ]:
import math

def atencion(Q, K, V):
    """Self-attention de producto punto escalado, desde cero.

    Q, K, V: (batch, tokens, dim)
    Devuelve: (salida, pesos_de_atención)
    """
    d_k = Q.size(-1)

    # TODO 1 — puntajes de similitud: cada query contra cada key, escalado por sqrt(d_k).
    #          Pista: K.transpose(-2, -1) intercambia los dos últimos ejes.
    scores = None

    # TODO 2 — conviertan los puntajes en pesos que sumen 1. ¿Sobre qué eje?
    pesos = None

    # TODO 3 — usen los pesos para promediar V.
    salida = None

    return salida, pesos

In [ ]:
# Verificación: ¿coincide con la implementación oficial de PyTorch?
salida, pesos = atencion(Q, K, V)
oficial = F.scaled_dot_product_attention(Q, K, V)

assert salida is not None, 'Todavía faltan los TODO.'
print('¿Los pesos suman 1 por fila?', torch.allclose(pesos.sum(-1), torch.ones(1, L)))
print('¿Coincide con PyTorch?      ', torch.allclose(salida, oficial, atol=1e-6))
print('\nError máximo:', (salida - oficial).abs().max().item())

<details><summary>🔑 Solución — ábranla solo si llevan más de diez minutos atascados</summary>

```python
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
pesos  = torch.softmax(scores, dim=-1)
salida = pesos @ V
```

Sobre el eje del softmax: `dim=-1` es la respuesta. Cada **fila** de la matriz de puntajes es un token mirando a todos los demás, y queremos que *esa* distribución sume 1. Si ponen `dim=-2` normalizan por columnas — o sea, cada token repartiendo su relevancia hacia afuera. No es un error de sintaxis: corre perfecto y calcula otra cosa. **Es el bug que no explota**, del que hablamos en la Sesión 1.
</details>

### Ahora sobre una frase de verdad

Los pesos de atención no son un número abstracto: son una matriz que se puede mirar.

In [ ]:
FRASE = ['el', 'gato', 'negro', 'duerme']
L2 = len(FRASE)

# Un embedding por token (aleatorio y con semilla: hoy nos importa la mecánica, no el significado)
torch.manual_seed(7)
X = torch.randn(1, L2, 16)

# En un transformer real, Q, K y V salen de proyectar X con matrices aprendidas.
# Hoy usamos X directamente: self-attention en su forma más desnuda.
_, W = atencion(X, X, X)

pd.DataFrame(W[0].detach().numpy(), index=FRASE, columns=FRASE).style \
  .background_gradient(cmap='Blues', axis=1) \
  .format('{:.2f}') \
  .set_caption('Cada FILA es un token repartiendo su atención. Cada fila suma 1.')

> 👀 **Lean la matriz por filas.** La fila de `gato` dice: de toda mi atención, cuánta le doy a `el`, cuánta a mí mismo, cuánta a `negro`, cuánta a `duerme`.
>
> Con embeddings aleatorios y sin entrenar, los pesos no significan nada — la diagonal domina simplemente porque un vector se parece a sí mismo. **Eso no es un defecto de la demo: es el punto.** La estructura ya existe; lo que falta es el entrenamiento que le enseñe a qué prestarle atención.
>
> En la Sesión 3 vamos a ver esta misma matriz sobre un modelo entrenado, y ahí sí van a ver `duerme` mirando a `gato` porque uno es el sujeto del otro.

<details><summary>🛟 Resultado esperado</summary>
Una matriz 4×4 con la diagonal más oscura, filas que suman 1, y ningún patrón lingüístico reconocible.
</details>

---
### 🔄 PUNTO DE SINCRONIZACIÓN — fin del Lab B

Todos deberían tener `¿Coincide con PyTorch? True`.

☕ **Pausa de 15 minutos.** Volvemos directo al Lab C.

---
# LAB C · La atención no sabe de orden
### ⏱ 20 minutos

Acaban de construir self-attention y verificaron que está bien.

Ahora la voy a romper.

Miren estas dos frases:

> **El perro mordió al hombre.**
> **El hombre mordió al perro.**

Mismas palabras. Significados opuestos. Una es un martes cualquiera; la otra es noticia.

**Pregunta:** la atención que ustedes escribieron, ¿las distingue?

Antes de correr nada — **voten**. Sí o no.

In [ ]:
# Las dos frases: mismos tokens, distinto orden
FRASE_A = ['el', 'perro', 'mordió', 'al', 'hombre']
FRASE_B = ['el', 'hombre', 'mordió', 'al', 'perro']

# Una tabla de embeddings: cada palabra tiene SIEMPRE el mismo vector
VOCAB = ['el', 'perro', 'mordió', 'al', 'hombre']
torch.manual_seed(0)
TABLA = {w: torch.randn(16) for w in VOCAB}

def embeber(frase):
    return torch.stack([TABLA[w] for w in frase]).unsqueeze(0)   # (1, L, 16)

XA, XB = embeber(FRASE_A), embeber(FRASE_B)
print('A:', ' '.join(FRASE_A), '→', tuple(XA.shape))
print('B:', ' '.join(FRASE_B), '→', tuple(XB.shape))

In [ ]:
# Pasamos las dos frases por SU atención
outA, _ = atencion(XA, XA, XA)
outB, _ = atencion(XB, XB, XB)

# ¿Qué vector le salió a 'perro' en cada frase?
# En A está en la posición 1. En B, en la posición 4.
vec_perro_A = outA[0, FRASE_A.index('perro')]
vec_perro_B = outB[0, FRASE_B.index('perro')]

print('¿El vector de "perro" es IDÉNTICO en las dos frases?')
print('   →', torch.allclose(vec_perro_A, vec_perro_B, atol=1e-6))
print('\nDiferencia máxima:', (vec_perro_A - vec_perro_B).abs().max().item())

> 😐 **`True`.**
>
> Para su atención, `perro` es exactamente la misma cosa cuando muerde que cuando lo muerden. La posición no entró nunca en la ecuación — vuelvan a mirar sus tres líneas y busquen dónde aparece el orden.
>
> No está. `softmax(QKᵀ/√d)·V` no tiene ni un solo término que dependa de dónde está cada token.
>
> **La self-attention pura es una bolsa de palabras.** Una bolsa muy sofisticada, que pesa cada palabra según su relación con las demás — pero una bolsa. El orden se le escurre.

Formalmente esto se llama **equivarianza a permutaciones**: si permutan la entrada, la salida se permuta igual, y el vector de cada token no cambia. Comprobémoslo en general, no solo para `perro`:

In [ ]:
# La frase B es la frase A con las posiciones 1 y 4 intercambiadas.
perm = [FRASE_A.index(w) if FRASE_A.count(w) == 1 else None for w in FRASE_B]
print('Permutación A→B:', perm)

# Si la atención es equivariante, permutar la salida de A debe dar la salida de B
outA_permutada = outA[0, perm]
print('\n¿atención(permutar(X)) == permutar(atención(X))?')
print('   →', torch.allclose(outA_permutada, outB[0], atol=1e-6))

### El arreglo

Si la atención no ve la posición, hay que **metérsela en el vector**.

La idea de *Attention Is All You Need*: sumarle a cada embedding una firma que dependa únicamente de dónde está el token. Ondas de seno y coseno de distintas frecuencias — cada posición recibe un patrón único, y posiciones cercanas reciben patrones parecidos.

In [ ]:
def codificacion_posicional(L, D):
    """Codificación posicional sinusoidal (Vaswani et al., 2017, sección 3.5)."""
    pos = torch.arange(L).unsqueeze(1).float()             # (L, 1)
    i   = torch.arange(0, D, 2).float()                    # (D/2,)
    div = torch.exp(-math.log(10000.0) * i / D)
    pe  = torch.zeros(L, D)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe.unsqueeze(0)                                 # (1, L, D)

PE = codificacion_posicional(5, 16)

pd.DataFrame(PE[0].numpy()).style \
  .background_gradient(cmap='RdBu', axis=None) \
  .format('{:.2f}') \
  .set_caption('Cada FILA es una posición. Cada posición tiene su huella digital.')

In [ ]:
# La misma atención de antes, pero ahora sumando la posición al embedding
outA_pe, _ = atencion(XA + PE, XA + PE, XA + PE)
outB_pe, _ = atencion(XB + PE, XB + PE, XB + PE)

vec_perro_A_pe = outA_pe[0, FRASE_A.index('perro')]
vec_perro_B_pe = outB_pe[0, FRASE_B.index('perro')]

print('¿El vector de "perro" sigue siendo idéntico?')
print('   →', torch.allclose(vec_perro_A_pe, vec_perro_B_pe, atol=1e-6))
print('\nDiferencia máxima ahora:', (vec_perro_A_pe - vec_perro_B_pe).abs().max().item())

> ✅ **`False`.** Y ahí está.
>
> Una suma. Eso fue todo lo que hizo falta para que el modelo distinga quién muerde a quién.
>
> Y noten el orden en que pasaron las cosas hoy — es el orden en que pasan siempre:
>
> 1. Construimos algo que funcionaba.
> 2. Descubrimos que estaba roto de una forma que no se veía.
> 3. Lo arreglamos.
> 4. El arreglo abre preguntas nuevas.
>
> **La pregunta nueva:** ¿por qué senos y cosenos? ¿Por qué no simplemente sumarle el número de la posición: 1, 2, 3, 4? ¿Y por qué los modelos de hoy —LLaMA, Qwen, el que corrieron la semana pasada— ya no usan esto sino algo llamado **RoPE**?
>
> Eso es lo que sigue, en las diapositivas.

<details><summary>🛟 Resultado esperado</summary>

Sin codificación posicional: `True` (el vector de `perro` es idéntico en las dos frases) y la equivarianza se confirma.
Con codificación posicional: `False`, con una diferencia máxima claramente distinta de cero.
</details>

---
## Cierre

Hoy escribieron el mecanismo central de todos los modelos que vamos a usar el resto del semestre. Tres líneas.

Lo que se llevan:

- **El tokenizador tiene opiniones,** y algunas les cuestan plata. Su proyecto es en español; ya saben qué significa eso.
- **La atención es un promedio ponderado** donde los pesos salen de una similitud. Nada más. Lo demás es escala y plomería.
- **La atención no sabe de orden.** Alguien tuvo que sumársela a mano, y esa decisión sigue evolucionando hasta hoy.

### Tarea para S03

1. Leer la **sección 3** de *Attention Is All You Need* — arxiv.org/abs/1706.03762. Ya escribieron la ecuación 1; ahora la van a reconocer en el paper. Van a leerla distinto de como la habrían leído ayer.
2. Su equipo llega a S03 con un **candidato a modelo base** para M1. Criterio nuevo, cortesía del Lab A: miren cómo tokeniza su dominio antes de enamorarse de un modelo.

